In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [7]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\ameya\AppData\Local\Temp\ipykernel_20348\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
%pwd

'd:\\learning\\projects\\AI\\langchain-ai\\research'

In [9]:
%cd ..

d:\learning\projects\AI\langchain-ai


c:\Users\ameya\anaconda3\envs\interview\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [10]:
%pwd

'd:\\learning\\projects\\AI\\langchain-ai'

Reading the PDF

In [11]:
file_path = "data/SDG.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()

In [12]:
data

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2020-09-15T15:08:34-04:00', 'author': 'Susana.Castillo', 'moddate': '2020-09-15T15:08:34-04:00', 'title': 'sdgs_targets_overview_resource.pdf', 'source': 'data/SDG.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}, page_content='4th SDG Youth Summer Camp – SDG Resource Document \nThe 2030 Agenda for Sustainable Development’s 17 \nSustainable Development Goals (SDGs)   \nGoal: This document enables 4th SDG Youth Summer Camp participants to i) get to \nknow the 17 SDGs, ii) explore what areas each goal covers under its targets, iii) identify \ntargets of most interest to participants, and iv) identify synergies between the SDGs \nand chosen target(s).  \n \n \nGoal 1. End poverty in all its forms everywhere  \nTarget 1.1 By 2030, eradicate extreme poverty for all people everywhere, currently measured \nas people living on less than $1.25 a day  \nTarget 1.2 By 2030, reduce at least by hal

In [13]:
len(data)

19

Extracting data from the pdf

In [14]:
question_gen = ""
for page in data:
    question_gen += page.page_content

In [15]:
question_gen

'4th SDG Youth Summer Camp – SDG Resource Document \nThe 2030 Agenda for Sustainable Development’s 17 \nSustainable Development Goals (SDGs)   \nGoal: This document enables 4th SDG Youth Summer Camp participants to i) get to \nknow the 17 SDGs, ii) explore what areas each goal covers under its targets, iii) identify \ntargets of most interest to participants, and iv) identify synergies between the SDGs \nand chosen target(s).  \n \n \nGoal 1. End poverty in all its forms everywhere  \nTarget 1.1 By 2030, eradicate extreme poverty for all people everywhere, currently measured \nas people living on less than $1.25 a day  \nTarget 1.2 By 2030, reduce at least by half the proportion of men, women and children of all \nages living in poverty in all its dimensions according to national definitions  \nTarget 1.3 Implement nationally appropriate social protection systems and measures for \nall, including floors, and by 2030 achieve substantial coverage of the poor and the vulnerable  \nTarget 

In [17]:
from langchain_text_splitters import TokenTextSplitter

Defining the text splitter

In [19]:
splitter_ques_gen = TokenTextSplitter(
    model_name="gpt-5-nano",
    chunk_size=10000,
    chunk_overlap=200,
)

Converting text in chunks

In [20]:
chunk_ques_gen = splitter_ques_gen.split_text(question_gen)

In [21]:
chunk_ques_gen

['4th SDG Youth Summer Camp – SDG Resource Document \nThe 2030 Agenda for Sustainable Development’s 17 \nSustainable Development Goals (SDGs)   \nGoal: This document enables 4th SDG Youth Summer Camp participants to i) get to \nknow the 17 SDGs, ii) explore what areas each goal covers under its targets, iii) identify \ntargets of most interest to participants, and iv) identify synergies between the SDGs \nand chosen target(s).  \n \n \nGoal 1. End poverty in all its forms everywhere  \nTarget 1.1 By 2030, eradicate extreme poverty for all people everywhere, currently measured \nas people living on less than $1.25 a day  \nTarget 1.2 By 2030, reduce at least by half the proportion of men, women and children of all \nages living in poverty in all its dimensions according to national definitions  \nTarget 1.3 Implement nationally appropriate social protection systems and measures for \nall, including floors, and by 2030 achieve substantial coverage of the poor and the vulnerable  \nTarget

In [23]:
len(chunk_ques_gen)

1

Converting the chunks to documents

In [26]:
from langchain_core.documents import Document

In [27]:
document_ques_gen = [Document(page_content = t) for t in chunk_ques_gen]

In [29]:
type(document_ques_gen[0])

langchain_core.documents.base.Document

In [30]:
splitter_ques_gen = TokenTextSplitter(
    model_name="gpt-5-nano",
    chunk_size=1000,
    chunk_overlap=100,
)

In [32]:
document_answer_gen = splitter_ques_gen.split_documents(document_ques_gen)

In [33]:
document_answer_gen

[Document(metadata={}, page_content='4th SDG Youth Summer Camp – SDG Resource Document \nThe 2030 Agenda for Sustainable Development’s 17 \nSustainable Development Goals (SDGs)   \nGoal: This document enables 4th SDG Youth Summer Camp participants to i) get to \nknow the 17 SDGs, ii) explore what areas each goal covers under its targets, iii) identify \ntargets of most interest to participants, and iv) identify synergies between the SDGs \nand chosen target(s).  \n \n \nGoal 1. End poverty in all its forms everywhere  \nTarget 1.1 By 2030, eradicate extreme poverty for all people everywhere, currently measured \nas people living on less than $1.25 a day  \nTarget 1.2 By 2030, reduce at least by half the proportion of men, women and children of all \nages living in poverty in all its dimensions according to national definitions  \nTarget 1.3 Implement nationally appropriate social protection systems and measures for \nall, including floors, and by 2030 achieve substantial coverage of th

Setting up LLM

In [36]:
from langchain_openai import ChatOpenAI


In [37]:
llm = ChatOpenAI(model="gpt-5-nano",temperature=1)

In [38]:
prompt_template = """
You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

------------
{text}
------------

Create questions that will prepare the coders or programmers for their tests.
Make sure not to lose any important information.

QUESTIONS:
"""

In [40]:
from langchain_core.prompts import PromptTemplate

In [41]:
PROMPT_QUESTIONS = PromptTemplate(template=prompt_template, input_variables=["text"])

2nd layer verification so we dont go off topic

In [42]:
refine_template = ("""
You are an expert at creating practice questions based on coding material and documentation.
Your goal is to help a coder or programmer prepare for a coding test.
We have received some practice questions to a certain extent: {existing_answer}.
We have the option to refine the existing questions or add new ones.
(only if necessary) with some more context below.
------------
{text}
------------

Given the new context, refine the original questions in English.
If the context is not helpful, please provide the original questions.
QUESTIONS:
"""
)

In [43]:
REFINE_PROMPT_QUESTIONS = PromptTemplate(input_variables=["existing_answer", "text"], template=refine_template)

In [45]:
from langchain_classic.chains.summarize import load_summarize_chain

Creating chain

In [47]:
ques_gen_chain = load_summarize_chain(llm,
    chain_type="refine",
    verbose=True,
    question_prompt=PROMPT_QUESTIONS,
    refine_prompt=REFINE_PROMPT_QUESTIONS,
)


Generating questions

In [48]:
ques = ques_gen_chain.run(document_ques_gen)
print(ques)

C:\Users\ameya\AppData\Local\Temp\ipykernel_20348\3052573051.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  ques = ques_gen_chain.run(document_ques_gen)




> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

------------
4th SDG Youth Summer Camp – SDG Resource Document 
The 2030 Agenda for Sustainable Development’s 17 
Sustainable Development Goals (SDGs)   
Goal: This document enables 4th SDG Youth Summer Camp participants to i) get to 
know the 17 SDGs, ii) explore what areas each goal covers under its targets, iii) identify 
targets of most interest to participants, and iv) identify synergies between the SDGs 
and chosen target(s).  
 
 
Goal 1. End poverty in all its forms everywhere  
Target 1.1 By 2030, eradicate extreme poverty for all people everywhere, currently measured 
as people living on less than $1.25 a day  
Target 1.2 By 2030, reduce at least by half 

Creating embeddings

In [50]:
from langchain_openai import OpenAIEmbeddings

In [51]:
embeddings = OpenAIEmbeddings()

Vector DB

In [53]:
from langchain_community.vectorstores import FAISS

In [54]:
vector_store = FAISS.from_documents(document_answer_gen,embeddings)

LLM for Answer generation

In [55]:
llm_answer_gen = ChatOpenAI(model="gpt-5-nano",temperature=1)

In [58]:
ques_list = ques.split("\n")
ques_list

['QUESTIONS:',
 '',
 '1) How many Sustainable Development Goals (SDGs) are described in this document?',
 '',
 '2) What is the full title of Goal 3?',
 '',
 '3) Which target number under Goal 5 aims to “Eliminate all forms of violence against all women and girls in the public and private spheres, including trafficking and sexual and other types of exploitation”?',
 '',
 '4) Which target number under Goal 6 aims to achieve universal and equitable access to safe and affordable drinking water?',
 '',
 '5) Which target under Goal 3 aims to end preventable deaths of newborns and children under 5 years of age?',
 '',
 '6) Under Goal 4, which target seeks to substantially increase the number of youth and adults who have relevant skills, including technical and vocational skills, for employment?',
 '',
 '7) True or False: Target 9.b is about youth employment.',
 '',
 '8) Which target focuses on eliminating all harmful practices, such as child, early and forced marriage and female genital mutil

In [62]:
from langchain_classic.chains import RetrievalQA

In [63]:
answer_generation_chain = RetrievalQA.from_chain_type(llm=llm_answer_gen,
chain_type="stuff",
 retriever=vector_store.as_retriever())

In [65]:
for question in ques_list:
    print("Question: ", question)
    answer = answer_generation_chain.run(question)
    print("Answer: ", answer)
    print("-------------------------------------------------------")
    with open("answers.txt", "a") as f:
        f.write("Question: " + question + "\n")
        f.write("Answer: " + answer + "\n")
        f.write("-------------------------------------------------------\n")

Question:  QUESTIONS:
Answer:  Here are ready-to-use questions you can pull from the SDG Resource Document. If you share your audience, format (quiz, discussion, workshop), and how many you want, I can tailor these further.

A. Quick factual questions (based on the document’s content)
1) How many Sustainable Development Goals (SDGs) are covered in the 2030 Agenda?
2) What are the four learning aims of the 4th SDG Youth Summer Camp resource document? (i–iv)
3) Name Target 7.1 and Target 7.2 under Goal 7.
4) What does Target 6.1 Focus on?
5) Which Target aims to end all forms of discrimination against women and girls?
6) What is Target 4.7 about?
7) Name a target from Goal 2 that relates to sustainable agriculture or food production.
8) Which target explicitly mentions reducing food waste and losses along the supply chain?
9) What does Target 13.2 ask countries to do?
10) Which target focuses on protecting and restoring water-related ecosystems by 2020?
11) What is Target 11.7 about?
12)